In [6]:
import pandas as pd
import numpy as np

# Merging the Data

In [7]:
SteamReviewsData = pd.read_csv('Data1-steam.csv')
VideoGamesData = pd.read_csv('Data2-videoGames.csv')

# print(SteamReviewsData.to_string()) 
# print(VideoGamesData.to_string()) 

VideoGamesData["match_key"] = VideoGamesData["Name"].str.strip().str.lower()
SteamReviewsData["match_key"] = SteamReviewsData["name"].str.strip().str.lower()

ColumnsSteamReviewsData = SteamReviewsData[["match_key", "genres", "positive_ratings", "negative_ratings"]].copy()

ColumnsSteamReviewsData["steam_total_reviews"] = ColumnsSteamReviewsData["positive_ratings"] + ColumnsSteamReviewsData["negative_ratings"]

ColumnsSteamReviewsData["steam_positive_review_ratio"] = ColumnsSteamReviewsData["positive_ratings"] / ColumnsSteamReviewsData["steam_total_reviews"]

ColumnsSteamReviewsData.drop(columns=["positive_ratings", "negative_ratings"], inplace=True)

MergedData = pd.merge(VideoGamesData, ColumnsSteamReviewsData, how='left', on='match_key')

MergedData.drop(columns=["match_key"], inplace=True)
MergedData.to_csv('MergedData.csv', index=False)

# print(MergedData.to_string()) 

# Cleaning and Imputation of Missing Data

### Cleaning and preparing player reviews

In [8]:
#round the steam score to 1 decimal place and scale it up so its values are betwwen 0 and 10
MergedData['steam_score'] = (MergedData['steam_positive_review_ratio'] * 10).round(1)

Producting single player score
* Scenario 1: Only User_Score available -> use it 
* Scenario 2: Only steam_score available -> use it 
* Scenario 3: Both available -> get standard weighted average 
    * I found out which kind of average I need using Gemini AI with this prompt: I have a pandas .cvs file where some of my User_Count column is low in number (max ~8,715), while my steam_total_reviews column is extremely high in number (30k+ is common). I need to calculate a weighted average of User_Score and steam_score. What kind of average do i need?

In [9]:
#scenarios 1 and 2
rowHasSteamScore = MergedData['steam_score'].notna()
rowHasUserScore = MergedData['User_Score'].notna()


#scenario 3
MergedData['Weight_User_Count'] = np.log1p(MergedData['User_Count']) #e.g. 8000 becomes 9.0
MergedData['Weight_Steam_Reviews'] = np.log1p(MergedData['steam_total_reviews']) #e.g. 35000 becomes 10.5

MergedData['total_u_score_weight'] = MergedData['Weight_User_Count'] + MergedData['Weight_Steam_Reviews']

# avg
MergedData['transformed_user_score'] = ((MergedData['User_Score'] * MergedData['Weight_User_Count']) +  (MergedData['steam_score'] * MergedData['Weight_Steam_Reviews'])) / MergedData['total_u_score_weight']



MergedData['final_score'] = MergedData['final_score'].round(1)
MergedData = MergedData.drop(columns=['Weight_User_Count', 'Weight_Steam_Reviews', 'total_u_score_weight'])


KeyError: 'final_score'